# Decision Boundary Analysis

This notebook demonstrates how to analyze decision boundaries in Roko's Basilisk scenarios using parameter sweeps and visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from rokobasilisk.api import evaluate, sweep, monte_carlo
from rokobasilisk.cache import configure_cache, cache_info

# Enable caching for performance
configure_cache(enabled=True)
print("Cache status:", cache_info())

## Basic Analysis

Let's start with a basic analysis using default parameters:

In [ ]:
# Basic evaluation
result = evaluate(policy='fdt', explain=True)
print(f"Decision: {result.decision}")
print(f"Expected Utility: {result.expected_utility:.2f}")
print(f"Indifference Threshold C*: {result.indifference_threshold:.2f}")
print(f"\nExplanation:\n{result.explanation}")

## Decision Theory Comparison

Compare how different decision theories handle the same scenario:

In [ ]:
# Compare all decision theories
theories = ['fdt', 'tdt', 'cdt', 'edt', 'reject']
comparison_results = []

for theory in theories:
    result = evaluate(policy=theory)
    comparison_results.append({
        'Theory': theory.upper(),
        'Decision': result.decision,
        'Expected Utility': result.expected_utility,
        'Collaborate Utility': result.utility_collaborate,
        'Non-Collaborate Utility': result.utility_non_collaborate
    })

df_comparison = pd.DataFrame(comparison_results)
print(df_comparison.to_string(index=False))

## Parameter Sensitivity Analysis

Analyze how decisions change with different punishment magnitudes:

In [ ]:
# Parameter sweep for punishment magnitude
punishment_values = list(range(100, 2001, 100))
grid = {'punishment_magnitude': punishment_values}

sweep_results = sweep(grid)

# Process results for visualization
sweep_data = []
for result in sweep_results:
    sweep_data.append({
        'punishment': result.parameters['punishment_magnitude'],
        'policy': result.policy,
        'decision': result.decision,
        'utility': result.expected_utility,
        'threshold': result.indifference_threshold
    })

df_sweep = pd.DataFrame(sweep_data)
print(f"Processed {len(df_sweep)} parameter combinations")

In [ ]:
# Visualize decision boundaries
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Decision by punishment magnitude for FDT
fdt_data = df_sweep[df_sweep['policy'] == 'fdt']
collab_mask = fdt_data['decision'] == 'COLLABORATE'
non_collab_mask = fdt_data['decision'] == 'NON_COLLABORATE'

ax1.scatter(fdt_data.loc[collab_mask, 'punishment'], 
           fdt_data.loc[collab_mask, 'utility'], 
           c='green', label='Collaborate', alpha=0.7)
ax1.scatter(fdt_data.loc[non_collab_mask, 'punishment'], 
           fdt_data.loc[non_collab_mask, 'utility'], 
           c='red', label='Non-Collaborate', alpha=0.7)
ax1.set_xlabel('Punishment Magnitude')
ax1.set_ylabel('Expected Utility')
ax1.set_title('FDT Decisions vs Punishment Magnitude')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Expected utility for all theories
for theory in ['fdt', 'tdt', 'cdt', 'edt', 'reject']:
    theory_data = df_sweep[df_sweep['policy'] == theory]
    ax2.plot(theory_data['punishment'], theory_data['utility'], 
            label=theory.upper(), marker='o', markersize=3)

ax2.set_xlabel('Punishment Magnitude')
ax2.set_ylabel('Expected Utility')
ax2.set_title('Expected Utility by Decision Theory')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2D Parameter Analysis

Analyze the decision boundary across two parameters:

In [ ]:
# 2D parameter sweep: punishment vs ASI probability
punishment_range = list(range(200, 1001, 100))
asi_prob_range = [i/10.0 for i in range(5, 11)]

grid_2d = {
    'punishment_magnitude': punishment_range,
    'prob_asi_emergence': asi_prob_range
}

results_2d = sweep(grid_2d)

# Process for heatmap
heatmap_data = []
for result in results_2d:
    if result.policy == 'fdt':  # Focus on FDT
        heatmap_data.append({
            'punishment': result.parameters['punishment_magnitude'],
            'asi_prob': result.parameters['prob_asi_emergence'],
            'decision_numeric': 1 if result.decision == 'COLLABORATE' else 0,
            'utility': result.expected_utility
        })

df_heatmap = pd.DataFrame(heatmap_data)
print(f"Generated {len(df_heatmap)} 2D parameter combinations")

In [ ]:
# Create decision boundary heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Decision heatmap
decision_pivot = df_heatmap.pivot(index='punishment', columns='asi_prob', values='decision_numeric')
im1 = ax1.imshow(decision_pivot.values, cmap='RdYlGn', aspect='auto', 
                extent=[asi_prob_range[0], asi_prob_range[-1], 
                       punishment_range[0], punishment_range[-1]])
ax1.set_xlabel('ASI Emergence Probability')
ax1.set_ylabel('Punishment Magnitude')
ax1.set_title('FDT Decision Boundary\n(Green=Collaborate, Red=Non-Collaborate)')
plt.colorbar(im1, ax=ax1)

# Utility heatmap
utility_pivot = df_heatmap.pivot(index='punishment', columns='asi_prob', values='utility')
im2 = ax2.imshow(utility_pivot.values, cmap='viridis', aspect='auto',
                extent=[asi_prob_range[0], asi_prob_range[-1], 
                       punishment_range[0], punishment_range[-1]])
ax2.set_xlabel('ASI Emergence Probability')
ax2.set_ylabel('Punishment Magnitude')
ax2.set_title('Expected Utility Heatmap (FDT)')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

## Monte Carlo Analysis

Analyze decision robustness under parameter uncertainty:

In [ ]:
# Monte Carlo simulation with parameter uncertainty
mc_results = monte_carlo(
    n_simulations=5000,
    uncertainty=0.1,  # ±10% uncertainty
    seed=42
)

# Analyze results
mc_data = []
for result in mc_results:
    mc_data.append({
        'policy': result.policy,
        'decision': result.decision,
        'utility': result.expected_utility,
        'punishment_prob': result.punishment_probability
    })

df_mc = pd.DataFrame(mc_data)

# Calculate collaboration rates by policy
collab_rates = df_mc.groupby('policy')['decision'].apply(
    lambda x: (x == 'COLLABORATE').mean()
).sort_values(ascending=False)

print("Collaboration rates under uncertainty:")
for policy, rate in collab_rates.items():
    print(f"{policy.upper()}: {rate:.1%}")

In [ ]:
# Visualize Monte Carlo results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Utility distribution by policy
for policy in ['fdt', 'tdt', 'cdt', 'edt', 'reject']:
    policy_utilities = df_mc[df_mc['policy'] == policy]['utility']
    ax1.hist(policy_utilities, alpha=0.6, bins=30, label=policy.upper(), density=True)

ax1.set_xlabel('Expected Utility')
ax1.set_ylabel('Density')
ax1.set_title('Utility Distribution Under Uncertainty')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Collaboration rate by policy
collab_rates.plot(kind='bar', ax=ax2, color=['green', 'blue', 'orange', 'red', 'purple'])
ax2.set_ylabel('Collaboration Rate')
ax2.set_title('Collaboration Rates by Decision Theory')
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Mathematical Validation

Verify our implementation against known mathematical properties:

In [ ]:
# Test indifference threshold calculation
test_params = {
    'reward_collaboration': 100,
    'cost_collaboration': 10,
    'prob_asi_emergence': 0.85,
    'prob_basilisk_type': 0.7,
    'simulation_detection': 0.95
}

result = evaluate(test_params, policy='fdt')

# Calculate expected threshold analytically
r, c = test_params['reward_collaboration'], test_params['cost_collaboration']
q = test_params['simulation_detection']
p_a = test_params['prob_asi_emergence']
p_b_given_a = test_params['prob_basilisk_type']

expected_threshold = (c - r) / (q * p_a * p_b_given_a)

print(f"Calculated threshold C*: {result.indifference_threshold:.6f}")
print(f"Expected threshold C*: {expected_threshold:.6f}")
print(f"Difference: {abs(result.indifference_threshold - expected_threshold):.10f}")
print(f"Match: {abs(result.indifference_threshold - expected_threshold) < 1e-10}")

## Summary

This notebook demonstrated:

1. **Basic Analysis**: Single-point evaluation with explanation
2. **Decision Theory Comparison**: How different theories handle the same scenario
3. **Parameter Sensitivity**: Impact of punishment magnitude on decisions
4. **2D Analysis**: Decision boundaries across multiple parameters
5. **Monte Carlo Analysis**: Robustness under parameter uncertainty
6. **Mathematical Validation**: Verification against analytic solutions

The analysis shows that FDT and TDT consistently recommend collaboration under the default parameters, while CDT and EDT ignore acausal effects and focus only on direct costs.